In [ ]:
# CSBP441 Applied Computer Vision
# Week 2 - LN04 Colab Notebook Code
#
# Topic:
#   Camera models, image formation, perspective projection, homogeneous
#   coordinates, intrinsic/extrinsic parameters, and distortion.
#
# Purpose:
#   Students first solve camera projection examples by hand, then use Python
#   to visualize projection, depth, vanishing points, camera intrinsics, and
#   lens distortion.
#
# How to use in Google Colab:
#   1. Upload this .py file to Colab or copy the cells into a notebook.
#   2. Run from top to bottom.
#   3. Complete the hand-calculation answers before running the check cells.
#   4. Check the generated outputs/ folder.



# LN04 Colab - Camera Models and Image Formation

This notebook follows the course cycle:

**hand calculation -> Python/OpenCV implementation -> result analysis -> real-system interpretation**

## Learning Goals

By the end, you should be able to:

- explain the pinhole camera model;
- project simple 3D points onto a 2D image plane;
- interpret how depth changes image size;
- use homogeneous coordinates for projection;
- distinguish intrinsic and extrinsic camera parameters;
- visualize radial lens distortion and simple undistortion.



## Setup in Google Colab

OpenCV, NumPy, and Matplotlib are usually already installed in Colab.
Run the installation line only if imports fail.



In [ ]:
# Uncomment and run this line in Colab only if needed:
# !pip install opencv-python numpy matplotlib



In [ ]:
from pathlib import Path

import cv2
import math
import matplotlib.pyplot as plt
import numpy as np


OUTPUT_DIR = Path("outputs_ln04")
OUTPUT_DIR.mkdir(exist_ok=True)

print("OpenCV version:", cv2.__version__)
print("Output folder:", OUTPUT_DIR.resolve())




## Helper Functions



In [ ]:
def show_image_rgb(img_bgr, title="", figsize=(6, 4)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def show_bgr_image_grid(examples, output_path, ncols=3):
    """Show BGR OpenCV images in a grid and save the figure."""
    nrows = math.ceil(len(examples) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, (title, image) in zip(axes, examples):
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[len(examples):]:
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()


def setup_image_plane(ax, title, width=640, height=480):
    ax.set_title(title)
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.set_xlabel("image x coordinate")
    ax.set_ylabel("image y coordinate")


def project_points_pinhole(points_3d, f=800, cx=320, cy=240):
    """Project 3D camera-coordinate points using x=fX/Z+cx, y=fY/Z+cy."""
    points_3d = np.asarray(points_3d, dtype=float)
    X = points_3d[:, 0]
    Y = points_3d[:, 1]
    Z = points_3d[:, 2]
    x = f * X / Z + cx
    y = f * Y / Z + cy
    return np.column_stack([x, y])


def draw_projected_points(points_2d, labels, width=640, height=480):
    img = np.full((height, width, 3), 245, dtype=np.uint8)
    for x in range(0, width, 80):
        cv2.line(img, (x, 0), (x, height), (220, 220, 220), 1)
    for y in range(0, height, 80):
        cv2.line(img, (0, y), (width, y), (220, 220, 220), 1)
    cv2.circle(img, (width // 2, height // 2), 6, (0, 0, 0), -1)
    cv2.putText(img, "principal point", (width // 2 + 10, height // 2 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (40, 40, 40), 2)
    for (x, y), label in zip(points_2d, labels):
        cv2.circle(img, (int(round(x)), int(round(y))), 7, (0, 0, 255), -1)
        cv2.putText(img, label, (int(round(x)) + 10, int(round(y)) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 170), 2)
    return img


def load_real_image_example(width=640, height=480):
    """Load a real image for distortion examples, with Colab upload fallback."""
    # Matplotlib usually ships with this sample real photograph.
    try:
        from matplotlib.cbook import get_sample_data

        sample_path = get_sample_data("grace_hopper.jpg", asfileobj=False)
        real_rgb = plt.imread(sample_path)
        if real_rgb.ndim == 2:
            real_rgb = cv2.cvtColor(real_rgb, cv2.COLOR_GRAY2RGB)
        real_bgr = cv2.cvtColor(real_rgb[:, :, :3], cv2.COLOR_RGB2BGR)
        real_bgr = cv2.resize(real_bgr, (width, height))
        print("Using Matplotlib sample real image: grace_hopper.jpg")
        return real_bgr
    except Exception as exc:
        print("Matplotlib sample image was not available:", exc)

    # In Colab, allow the student to upload one real image.
    try:
        from google.colab import files  # type: ignore

        print("Upload one real image for the camera distortion example.")
        uploaded = files.upload()
        for file_name in uploaded:
            data = np.frombuffer(uploaded[file_name], np.uint8)
            img = cv2.imdecode(data, cv2.IMREAD_COLOR)
            if img is not None:
                print("Using uploaded real image:", file_name)
                return cv2.resize(img, (width, height))
    except Exception as exc:
        print("Upload not available or skipped:", exc)

    # Last fallback: create a photo-like scene. Prefer a real uploaded/sample
    # image when possible.
    fallback = np.full((height, width, 3), (210, 225, 235), dtype=np.uint8)
    cv2.rectangle(fallback, (60, 80), (580, 420), (190, 210, 215), -1)
    cv2.circle(fallback, (320, 210), 80, (80, 130, 190), -1)
    cv2.rectangle(fallback, (250, 290), (390, 430), (70, 90, 120), -1)
    cv2.putText(fallback, "Upload a real photo in Colab", (120, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (40, 40, 40), 2)
    print("Using fallback image. For a true real-image example, upload a photo in Colab.")
    return fallback




## Part A - Hand Calculation: Pinhole Projection

In a simple pinhole camera, a 3D point \((X,Y,Z)\) in camera coordinates
projects to:

x = fX/Z

y = fY/Z

Complete by hand first.

Given focal length `f = 2`, project these points:

P1 = (10, 6, 4)

P2 = (25, 15, 10)

Questions:

1. Compute x and y for P1.
2. Compute x and y for P2.
3. Why do these two points project to the same image coordinate?



In [ ]:
f_hand = 2
P1 = np.array([10, 6, 4])
P2 = np.array([25, 15, 10])

def simple_project(P, f):
    X, Y, Z = P
    return np.array([f * X / Z, f * Y / Z])

print("Check after solving by hand:")
print("P1 projects to:", simple_project(P1, f_hand))
print("P2 projects to:", simple_project(P2, f_hand))
print("Both lie on the same visual ray from the camera center.")




## Part B - Visualize Depth and Magnification

A point or object farther from the camera appears smaller because projection
divides by depth \(Z\).

We project the same square at different depths.



In [ ]:
square_near = np.array([
    [-1, -1, 4],
    [1, -1, 4],
    [1, 1, 4],
    [-1, 1, 4],
])

square_far = np.array([
    [-1, -1, 8],
    [1, -1, 8],
    [1, 1, 8],
    [-1, 1, 8],
])

near_2d = project_points_pinhole(square_near, f=800)
far_2d = project_points_pinhole(square_far, f=800)

fig, ax = plt.subplots(figsize=(7, 5))
setup_image_plane(ax, "Same 3D square at two depths")
ax.plot(*np.vstack([near_2d, near_2d[0]]).T, "o-", label="near: Z=4", linewidth=3)
ax.plot(*np.vstack([far_2d, far_2d[0]]).T, "o-", label="far: Z=8", linewidth=3)
ax.scatter([320], [240], c="black", s=50, label="principal point")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "depth_magnification_projection.png", dpi=150)
plt.show()

print("Near projected width:", abs(near_2d[1, 0] - near_2d[0, 0]))
print("Far projected width:", abs(far_2d[1, 0] - far_2d[0, 0]))




## Part C - Homogeneous Projection Matrix

Homogeneous coordinates allow projection to be written as matrix
multiplication, followed by division by the third coordinate.

Example projection matrix:

P = [[f, 0, cx, 0],
     [0, f, cy, 0],
     [0, 0,  1, 0]]

For a homogeneous 3D point:

Xh = [X, Y, Z, 1]

Compute:

xh = P @ Xh

Then convert to image coordinates:

x = xh[0] / xh[2]

y = xh[1] / xh[2]



In [ ]:
f = 800
cx, cy = 320, 240
P_matrix = np.array([
    [f, 0, cx, 0],
    [0, f, cy, 0],
    [0, 0, 1, 0],
], dtype=float)

Xh = np.array([1.5, 0.5, 5.0, 1.0])
xh = P_matrix @ Xh
pixel = xh[:2] / xh[2]

print("Projection matrix P:")
print(P_matrix)
print("Homogeneous image point:", xh)
print("Pixel coordinate after divide by w/Z:", pixel)

img_points = draw_projected_points(np.array([pixel]), ["X"], width=640, height=480)
cv2.imwrite(str(OUTPUT_DIR / "homogeneous_projection_point.png"), img_points)
show_image_rgb(img_points, "Projected Point from Homogeneous Matrix")




## Part D - Intrinsic Parameters

The intrinsic matrix describes the camera itself:

K = [[fx, s, cx],
     [0, fy, cy],
     [0,  0,  1]]

Meaning:

- fx, fy: focal length in pixel units;
- cx, cy: principal point;
- s: skew, usually 0 for modern cameras.

Question:

What changes in the image if we increase fx and fy?



In [ ]:
points_3d = np.array([
    [-1.0, -0.6, 5.0],
    [1.0, -0.6, 5.0],
    [1.0, 0.6, 5.0],
    [-1.0, 0.6, 5.0],
    [0.0, 0.0, 5.0],
])
labels = ["A", "B", "C", "D", "O"]

proj_f500 = project_points_pinhole(points_3d, f=500, cx=320, cy=240)
proj_f1000 = project_points_pinhole(points_3d, f=1000, cx=320, cy=240)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pts, title in [
    (axes[0], proj_f500, "f = 500 pixels"),
    (axes[1], proj_f1000, "f = 1000 pixels"),
]:
    setup_image_plane(ax, title)
    ax.plot(*np.vstack([pts[:4], pts[0]]).T, "o-", linewidth=3)
    for (x, y), label in zip(pts, labels):
        ax.text(x + 8, y - 8, label)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "intrinsic_focal_length_comparison.png", dpi=150)
plt.show()




## Part E - Extrinsic Parameters

Extrinsic parameters describe the camera pose relative to the world:

- R: rotation
- t: translation

The pipeline is:

world point -> camera coordinates -> projection -> pixel coordinates

In this simple example, we translate the scene along Z. Increasing Z moves the
object farther away.



In [ ]:
world_square = np.array([
    [-1, -1, 0],
    [1, -1, 0],
    [1, 1, 0],
    [-1, 1, 0],
])

def project_with_translation(points_world, tz, f=800, cx=320, cy=240):
    points_camera = points_world + np.array([0, 0, tz])
    return project_points_pinhole(points_camera, f=f, cx=cx, cy=cy)

proj_tz4 = project_with_translation(world_square, tz=4)
proj_tz8 = project_with_translation(world_square, tz=8)

fig, ax = plt.subplots(figsize=(7, 5))
setup_image_plane(ax, "Extrinsic translation: moving object farther")
ax.plot(*np.vstack([proj_tz4, proj_tz4[0]]).T, "o-", label="tz=4", linewidth=3)
ax.plot(*np.vstack([proj_tz8, proj_tz8[0]]).T, "o-", label="tz=8", linewidth=3)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "extrinsic_translation_depth.png", dpi=150)
plt.show()




## Part F - Vanishing Point Intuition

Parallel 3D lines can appear to meet at a vanishing point in the image. This
is one of the most visible effects of perspective projection.



In [ ]:
road = np.full((480, 640, 3), 245, dtype=np.uint8)

# Horizon.
cv2.line(road, (0, 160), (640, 160), (180, 180, 180), 2)
vanishing_point = (320, 160)

# Road boundaries and lane lines.
for x_bottom in [80, 220, 420, 560]:
    cv2.line(road, (x_bottom, 480), vanishing_point, (60, 60, 60), 2)
cv2.circle(road, vanishing_point, 7, (0, 0, 255), -1)
cv2.putText(road, "vanishing point", (335, 155), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 180), 2)
cv2.putText(road, "parallel 3D road lines", (30, 440), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (40, 40, 40), 2)

cv2.imwrite(str(OUTPUT_DIR / "vanishing_point_intuition.png"), road)
show_image_rgb(road, "Vanishing Point: Parallel 3D Lines Meet in the Image", figsize=(7, 5))




## Part G - Radial Distortion

Real lenses are not perfect. Radial distortion bends image locations depending
on distance from the image center.

Barrel distortion makes lines bend outward.

Pincushion distortion makes lines bend inward.



In [ ]:
def create_grid_image(width=640, height=480, step=40):
    img = np.full((height, width, 3), 245, dtype=np.uint8)
    for x in range(0, width, step):
        cv2.line(img, (x, 0), (x, height), (80, 80, 80), 1)
    for y in range(0, height, step):
        cv2.line(img, (0, y), (width, y), (80, 80, 80), 1)
    cv2.circle(img, (width // 2, height // 2), 5, (0, 0, 255), -1)
    return img


def apply_radial_distortion(img, k1):
    h, w = img.shape[:2]
    cx, cy = w / 2, h / 2
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    xn = (x - cx) / cx
    yn = (y - cy) / cy
    r2 = xn * xn + yn * yn
    factor = 1 + k1 * r2
    map_x = (cx + xn * factor * cx).astype(np.float32)
    map_y = (cy + yn * factor * cy).astype(np.float32)
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)


grid = create_grid_image()
real_image = load_real_image_example(width=640, height=480)

barrel = apply_radial_distortion(grid, k1=-0.35)
pincushion = apply_radial_distortion(grid, k1=0.25)
real_barrel = apply_radial_distortion(real_image, k1=-0.35)
real_pincushion = apply_radial_distortion(real_image, k1=0.25)

cv2.imwrite(str(OUTPUT_DIR / "grid_original.png"), grid)
cv2.imwrite(str(OUTPUT_DIR / "barrel_distortion.png"), barrel)
cv2.imwrite(str(OUTPUT_DIR / "pincushion_distortion.png"), pincushion)
cv2.imwrite(str(OUTPUT_DIR / "real_image_original.png"), real_image)
cv2.imwrite(str(OUTPUT_DIR / "real_image_barrel_distortion.png"), real_barrel)
cv2.imwrite(str(OUTPUT_DIR / "real_image_pincushion_distortion.png"), real_pincushion)

show_bgr_image_grid([
    ("Synthetic: Original Grid", grid),
    ("Synthetic: Barrel Distortion", barrel),
    ("Synthetic: Pincushion Distortion", pincushion),
    ("Real Image: Original", real_image),
    ("Real Image: Barrel Distortion", real_barrel),
    ("Real Image: Pincushion Distortion", real_pincushion),
], OUTPUT_DIR / "radial_distortion_comparison.png")




## Part H - Undistortion Idea

In real camera calibration, OpenCV estimates the camera matrix and distortion
coefficients. Then `cv2.undistort()` can correct lens distortion.

Important note:

The distorted image in Part G was created using our own simple synthetic
radial mapping. It was not produced by a real calibrated camera. Therefore,
`cv2.undistort()` with guessed coefficients may look almost unchanged or may
not properly reverse the distortion.

For teaching, the comparison below uses the inverse of the same synthetic
mapping so the correction is visible.



In [ ]:
h, w = grid.shape[:2]
K = np.array([
    [500, 0, w / 2],
    [0, 500, h / 2],
    [0, 0, 1],
], dtype=np.float32)

# Approximate correction for the synthetic barrel distortion created earlier.
# Since barrel used k1=-0.35, we use the opposite sign to visibly reduce it.
synthetic_corrected = apply_radial_distortion(barrel, k1=0.35)
real_corrected = apply_radial_distortion(real_barrel, k1=0.35)

cv2.imwrite(str(OUTPUT_DIR / "synthetic_corrected_from_barrel.png"), synthetic_corrected)
cv2.imwrite(str(OUTPUT_DIR / "real_image_corrected_from_barrel.png"), real_corrected)

show_bgr_image_grid([
    ("Synthetic: Original Grid", grid),
    ("Synthetic: Barrel Distortion", barrel),
    ("Synthetic: Approx. Corrected\nusing inverse mapping", synthetic_corrected),
    ("Real Image: Original", real_image),
    ("Real Image: Barrel Distortion", real_barrel),
    ("Real Image: Approx. Corrected\nusing inverse mapping", real_corrected),
], OUTPUT_DIR / "undistortion_idea_comparison.png")

print("Example camera matrix K:")
print(K)
print("Real lesson: undistortion works only when the correction model matches the distortion model.")




## Part I - Student Analysis Questions

Answer these in your submitted report.

1. In the hand example, why do P1 and P2 project to the same image point?
2. What happens to projected object size when depth Z increases?
3. Why does homogeneous projection require division by the third coordinate?
4. What is the difference between intrinsic and extrinsic camera parameters?
5. Which parameter changes the apparent zoom: focal length, principal point, or camera translation?
6. Why can parallel 3D lines meet in a 2D image?
7. Why does radial distortion matter for measurement tasks?
8. Give one real vision system where camera calibration is important. Explain the input, camera-model step, output, and decision/action.

## Submission Checklist

Submit:

- hand calculations for Parts A and C;
- screenshots or saved figures from the Colab output;
- answers to the analysis questions;
- one short real-system interpretation paragraph.



In [ ]:
print("Generated output files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)
